# Batch correction with totalVI (joint RNA + protein)

totalVI (Gayoso et al., *Nat Methods* 2021) is scvi-tools' joint generative model for CITE-seq / Total-seq data. It learns a shared latent space across RNA and ADT (antibody-derived tag) while explicitly modelling protein background, and corrects batch effects in both modalities simultaneously.

This is one of the **omicverse batch-correction zoo** tutorials. For an overview of all backends, the recommendation tree, and the unified `_BATCH_OBSM` schema, see [batch/index](../index.md). For the side-by-side comparison of all backends on the NeurIPS 2021 multi-batch benchmark, see [t_single_batch](../t_single_batch.ipynb).

Optional dependency: `pip install scvi-tools`.

**Input contract**: 
- raw RNA counts in `adata.layers['counts']`,
- raw ADT counts in `adata.obsm[<protein_expression_obsm_key>]` as a `(cell × protein)` matrix.

If you have a `MuData` object with paired modalities, totalVI's `setup_mudata` is recommended; the omicverse wrapper uses the AnnData path (`setup_anndata`) which is the older API.


## Load a multi-batch dataset

We use the same toy multi-batch AnnData as the other zoo tutorials — three NeurIPS 2021 batches concatenated. Replace with your own dataset by changing `adata` and `batch_key` below.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

# Replace these URLs with your own dataset; the three are
# the same multi-batch dataset used by t_single_batch.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.obs['batch'] = adata.obs['batch'].astype('category')
adata

## Preprocess + PCA (shared across all backends)

Every backend in the zoo starts from the same QC'd, log-normalised AnnData with `scaled|original|X_pca` in obsm. Read [t_single_batch](../t_single_batch.ipynb) for the full discussion of these steps.

In [ ]:
adata = ov.pp.qc(adata,
                 tresh={'mito_perc': 0.2, 'nUMIs': 500,
                        'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson',
                         n_HVGs=2000, batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## Run `ov.single.batch_correction(methods='totalVI')`

The wrapper routes method-specific kwargs to the right destination — for scvi-tools backends this includes splitting between `__init__` (architecture) and `.train()` (optimisation). See the **Key parameters** section below.

In [ ]:
model = ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='totalVI',
    # Required: the obsm slot with the (cell, protein) ADT matrix:
    protein_expression_obsm_key='protein_counts',
    # Architecture:
    n_latent=20,
    # Optimisation:
    max_epochs=400, batch_size=256, early_stopping=True,
)
model

## Visualise the corrected embedding

Every backend writes its corrected representation to a stable obsm key — for **totalVI (joint RNA + protein)** it is `adata.obsm['X_totalVI']`. We project it via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_totalvi'] = ov.utils.mde(adata.obsm['X_totalVI'])
ov.pl.embedding(
    adata,
    basis='X_mde_totalvi',
    color=['batch'],
    frameon='small',
    title='totalVI (joint RNA + protein) — coloured by batch',
)

## Key parameters

**Required**:
- `protein_expression_obsm_key` — name of the obsm slot with the protein matrix.

**Architecture** (routed to `scvi.model.TOTALVI.__init__`):
- `n_latent`, `gene_dispersion`, `protein_dispersion`, `gene_likelihood`, `latent_distribution`.

**Optimisation** (routed to `scvi.model.TOTALVI.train`):
- `max_epochs`, `batch_size`, `early_stopping`, `accelerator`.


## Related tutorials

- `scVI` — RNA-only version when you have no protein data.
- `scANVI` — RNA + semi-supervised labels (different problem shape).

For a side-by-side comparison of every backend on the same benchmark + scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).